# Znajdywanie kolorów chwytaka

## Tworzenie dataframe

In [1]:
import pandas as pd
import numpy as np
from PIL import Image
from pathlib import Path

In [2]:
SAVE_DIR = Path("outputs")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
masks_array = np.load(str(SAVE_DIR)+'/masks_array.npy')
incomplete_detect = np.load(str(SAVE_DIR)+'/incomplete_detect.npy')
df = pd.read_csv(str(SAVE_DIR)+'/df.csv')
incomplete_detect

array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  21,  28,  34,
        42,  49,  52,  57,  58,  59,  66,  67,  68,  70,  72,  76,  77,
        88,  89,  90,  91,  96, 103, 104, 106, 108, 111, 112, 113, 114,
       115, 116, 117, 118, 119, 120, 123, 152])

### (x,y) z GDSAM

In [7]:
xy_array = np.load(str(SAVE_DIR)+'/all_boxes.npy')
xy_array = xy_array.reshape(-1,12)
xy_array = np.delete(xy_array, incomplete_detect, axis=0)  # Usuń niekompletne detekcje
xy_array[15]

array([0.35889727, 0.57750672, 0.41108453, 0.63922018, 0.37848032,
       0.55425894, 0.45079881, 0.59179437, 0.40508613, 0.56294137,
       0.45511022, 0.62966877])

#### DLA OPENAI

In [8]:
import base64
from openai import OpenAI
import os

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

Kodowanie base64 do API openai

In [ ]:
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")
    
encoded_images_gpt = df["color_path"].apply(encode_image).to_numpy().reshape(-1,1)

encoded_images_gpt = np.delete(encoded_images_gpt, incomplete_detect, axis=0)
encoded_images_gpt.shape

(125, 1)

In [ ]:
xy_array_gpt = []
for i in range(0, encoded_images_gpt.shape[0]):
    base64_image = encoded_images_gpt[i][0]
    
    prompt_text = (
        "The image shows a robot gripper with three colored elements: pink, green, and blue. "
        "Your task is to detect each element and provide normalized bounding box coordinates "
        "in the exact format: (x_min,y_min,x_max,y_max),(x_min,y_min,x_max,y_max),(x_min,y_min,x_max,y_max) "
        "for pink, green, blue respectively. "
        "Coordinates must be floating-point numbers in [0,1] range, relative to image width and height "
        "(origin at top-left corner). "
        "Return ONLY the coordinates in this format, no additional text, explanations, or comments."
    )

    response = client.responses.create(
        model="gpt-5",
        input=[
            {
                "role": "user",
                "content": [
                    { "type": "input_text", "text": prompt_text },
                    {
                        "type": "input_image",
                        "image_url": f"data:image/jpeg;base64,{base64_image}",
                        "detail": "high"
                    },
                ],
            }
        ],
    )
    xy_array_gpt.append(response.output_text)

print(xy_array_gpt[:2])

['(0.4648,0.3514,0.4980,0.4509),(0.4961,0.3325,0.5273,0.3962),(0.5449,0.3443,0.5781,0.4387)', '(0.630,0.350,0.670,0.490),(0.650,0.300,0.690,0.380),(0.690,0.350,0.740,0.490)']


#### DLA GOOGLE

In [18]:
import dotenv
dotenv.load_dotenv()

True

In [19]:
import google.generativeai as genai
import os

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

models = [m.name for m in genai.list_models()
          if "generateContent" in getattr(m, "supported_generation_methods", [])]
print([m for m in models if "robot" in m.lower() or "er" in m.lower() or "1.5" in m.lower()])

# Zmieniono na gemini-2.0-flash - wyższe limity, brak rate limitingu
model = genai.GenerativeModel('models/gemini-2.0-flash')

['models/gemini-robotics-er-1.5-preview', 'models/gemini-2.5-computer-use-preview-10-2025']


In [20]:
encoded_images_gemini = df["color_path"].to_numpy().reshape(-1,1)

encoded_images_gemini = np.delete(encoded_images_gemini, incomplete_detect, axis=0)
encoded_images_gemini.shape

(125, 1)

#### KONWERSJA

In [22]:
import time
import traceback

xy_array_gemini = []
errors = []
checkpoint_file = str(SAVE_DIR) + '/xy_array_gemini_checkpoint.npy'

# Spróbuj wczytać istniejący checkpoint
try:
    xy_array_gemini = np.load(checkpoint_file, allow_pickle=True).tolist()
    start_idx = len(xy_array_gemini)
    print(f"Wznowienie od obrazu {start_idx}")
except FileNotFoundError:
    start_idx = 0
    print(f"Rozpoczynanie od początku")

for i in range(start_idx, encoded_images_gemini.shape[0]):
    try:
        image_path = encoded_images_gemini[i][0]
        img = Image.open(image_path)
        w, h = img.size

        prompt_text = (
            "The image shows a robot gripper with three colored elements: pink, green, and blue. "
            "Your task is to detect each element and provide normalized bounding box coordinates "
            "in the exact format: (x_min,y_min,x_max,y_max),(x_min,y_min,x_max,y_max),(x_min,y_min,x_max,y_max) "
            "for pink, green, blue respectively. "
            "Coordinates must be floating-point numbers in [0,1] range, relative to image width and height "
            "(origin at top-left corner). "
            "Return ONLY the coordinates in this format, no additional text, explanations, or comments."
        )

        resp = model.generate_content([prompt_text, img])
        xy_array_gemini.append(resp.text.strip())
        
        # Zapisz checkpoint co 5 obrazów
        if (i + 1) % 5 == 0:
            np.save(checkpoint_file, np.array(xy_array_gemini, dtype=object))
            print(f"Checkpoint: Przetworzono {i + 1}/{encoded_images_gemini.shape[0]} obrazów")
            time.sleep(1)  # Czekaj między batchami
            
    except Exception as e:
        error_msg = str(e)
        print(f"\n⚠️  Błąd dla obrazu {i}: {type(e).__name__}")
        
        # Obsługa rate limitingu
        if "429" in error_msg or "quota" in error_msg.lower() or "exceeded" in error_msg.lower():
            print("❌ LIMIT PRZEKROCZONY - Gemini API rate limit")
            
            # Szukaj czasu czekania w błędzie
            retry_after = 3600  # Default: 1 godzina
            if "retry_delay" in error_msg or "Retry in" in error_msg:
                import re
                match = re.search(r'(\d+\.?\d*)\s*s', error_msg)
                if match:
                    retry_after = float(match.group(1)) + 10
            
            minutes = int(retry_after // 60)
            seconds = int(retry_after % 60)
            print(f"⏳ Czekaj: {minutes}min {seconds}s (albo do jutra na nowy limit darmowy)")
            print(f"   Możesz: 1) Czekać, 2) Zmienić na płatny plan, 3) Użyć innego modelu")
            
            # Zapisz postęp
            np.save(checkpoint_file, np.array(xy_array_gemini, dtype=object))
            break  # Zatrzymaj pętlę
            
        else:
            # Inne błędy - dodaj pusty string i kontynuuj
            errors.append((i, error_msg[:100]))
            xy_array_gemini.append("")
            time.sleep(3)

# Zapisz ostateczny wynik
np.save(checkpoint_file, np.array(xy_array_gemini, dtype=object))
print(f"\n✅ Wyniki: {len(xy_array_gemini)}/{encoded_images_gemini.shape[0]}")
if errors:
    print(f"⚠️  Błędy dla {len(errors)} obrazów:")
    for idx, err in errors[:5]:
        print(f"   Obraz {idx}: {err}...")

print(xy_array_gemini[:2])

Wznowienie od obrazu 0

⚠️  Błąd dla obrazu 0: ResourceExhausted
❌ LIMIT PRZEKROCZONY - Gemini API rate limit
⏳ Czekaj: 0min 48s (albo do jutra na nowy limit darmowy)
   Możesz: 1) Czekać, 2) Zmienić na płatny plan, 3) Użyć innego modelu

✅ Wyniki: 0/125
[]


In [ ]:
xy_array_gemini = np.array(xy_array_gemini).reshape(-1,1)
xy_array_gemini.shape

(10, 1)

In [ ]:
xy_array_gemini[0]

array(['(0.39,0.23,0.43,0.28),(0.46,0.24,0.50,0.27),(0.53,0.28,0.57,0.32)'],
      dtype='<U77')

In [ ]:
#np.save(str(SAVE_DIR)+'/xy_array_gemini3.npy', xy_array_gemini)
#np.save(str(SAVE_DIR)+'/xy_array_gpt3.npy', xy_array_gpt)

#### IMPORT GOTOWYCH

In [4]:
xy_array_gpt = np.load(str(SAVE_DIR)+'/xy_array_gpt3.npy')
xy_array_gemini = np.load(str(SAVE_DIR)+'/xy_array_gemini2.npy')
xy_array = np.load(str(SAVE_DIR)+'/all_boxes.npy')
xy_array = xy_array.reshape(-1,12)  # Reshape do formatu (n, 12)
xy_array = np.delete(xy_array, incomplete_detect, axis=0)  # Usuń niekompletne detekcje

In [5]:
xy_array_gpt.shape

(125,)

In [18]:
import re

def parse_bbox_rows(arr):
    out = []
    for row in arr:
        text = row[0] if isinstance(row, (list, np.ndarray)) else str(row)
        boxes = re.findall(r'\(([0-9]*\.?[0-9]+)\s*,\s*([0-9]*\.?[0-9]+)\s*,\s*([0-9]*\.?[0-9]+)\s*,\s*([0-9]*\.?[0-9]+)\)', text)
        nums = [float(x) for b in boxes for x in b]
        # dociecie/padding do 12 liczb (3 boxy po 4)
        if len(nums) < 12:
            nums += [0.0] * (12 - len(nums))
        else:
            nums = nums[:12]
        out.append(nums)
    return np.array(out, dtype=float)

In [19]:
# Parsowanie bounding boxów zamiast pojedynczych punktów
xy_array_openai_bbox = parse_bbox_rows(xy_array_gpt)
xy_array_gemini_bbox = parse_bbox_rows(xy_array_gemini)

print("OpenAI bbox shape:", xy_array_openai_bbox.shape)
print("Gemini bbox shape:", xy_array_gemini_bbox.shape)
print("\nPrzykład OpenAI bbox (pierwszy wiersz):", xy_array_openai_bbox[0])
print("Przykład Gemini bbox (pierwszy wiersz):", xy_array_gemini_bbox[0])

OpenAI bbox shape: (125, 12)
Gemini bbox shape: (10, 12)

Przykład OpenAI bbox (pierwszy wiersz): [0.4648 0.3514 0.498  0.4509 0.4961 0.3325 0.5273 0.3962 0.5449 0.3443
 0.5781 0.4387]
Przykład Gemini bbox (pierwszy wiersz): [0.39 0.23 0.43 0.28 0.46 0.24 0.5  0.27 0.53 0.28 0.57 0.32]


## Wizualizacja

In [20]:
from pathlib import Path
import matplotlib.pyplot as plt
import cv2

In [21]:
id_array = np.delete(np.arange(172), incomplete_detect, axis=0)
id_array.shape

(125,)

### Wizualizacja z Bounding Boxami

In [ ]:
import matplotlib.patches as patches

# Utworzenie folderu na ploty z bbox
plots_bbox_dir = Path("plots_comparison/bbox")
plots_bbox_dir.mkdir(parents=True, exist_ok=True)

num_to_plot = min(10, len(xy_array), len(xy_array_openai_bbox), len(xy_array_gemini_bbox), len(id_array))

for nr_xy in range(num_to_plot):
    fig, axes = plt.subplots(1, 3, figsize=(36, 12))

    # Wczytaj obraz raz
    img_bgr = cv2.imread(df['color_path'][id_array[nr_xy]])
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]

    # 1) GDSAM (bounding boxy)
    axes[0].imshow(img_rgb)
    # Pink box
    x_min, y_min, x_max, y_max = xy_array[nr_xy][0:4]
    rect1 = patches.Rectangle((x_min*w, y_min*h), (x_max-x_min)*w, (y_max-y_min)*h, 
                               linewidth=2, edgecolor='magenta', facecolor='none', label='Pink')
    axes[0].add_patch(rect1)
    # Green box
    x_min, y_min, x_max, y_max = xy_array[nr_xy][4:8]
    rect2 = patches.Rectangle((x_min*w, y_min*h), (x_max-x_min)*w, (y_max-y_min)*h, 
                               linewidth=2, edgecolor='green', facecolor='none', label='Green')
    axes[0].add_patch(rect2)
    # Blue box
    x_min, y_min, x_max, y_max = xy_array[nr_xy][8:12]
    rect3 = patches.Rectangle((x_min*w, y_min*h), (x_max-x_min)*w, (y_max-y_min)*h, 
                               linewidth=2, edgecolor='blue', facecolor='none', label='Blue')
    axes[0].add_patch(rect3)
    axes[0].set_title(f'GDSAM (#{nr_xy})', fontsize=16)
    axes[0].legend()
    axes[0].axis('off')

    # 2) OpenAI (bounding boxy)
    axes[1].imshow(img_rgb)
    # Pink box
    x_min, y_min, x_max, y_max = xy_array_openai_bbox[nr_xy][0:4]
    rect1 = patches.Rectangle((x_min*w, y_min*h), (x_max-x_min)*w, (y_max-y_min)*h, 
                               linewidth=2, edgecolor='magenta', facecolor='none', label='Pink')
    axes[1].add_patch(rect1)
    # Green box
    x_min, y_min, x_max, y_max = xy_array_openai_bbox[nr_xy][4:8]
    rect2 = patches.Rectangle((x_min*w, y_min*h), (x_max-x_min)*w, (y_max-y_min)*h, 
                               linewidth=2, edgecolor='green', facecolor='none', label='Green')
    axes[1].add_patch(rect2)
    # Blue box
    x_min, y_min, x_max, y_max = xy_array_openai_bbox[nr_xy][8:12]
    rect3 = patches.Rectangle((x_min*w, y_min*h), (x_max-x_min)*w, (y_max-y_min)*h, 
                               linewidth=2, edgecolor='blue', facecolor='none', label='Blue')
    axes[1].add_patch(rect3)
    axes[1].set_title('OpenAI (BBox)', fontsize=16)
    axes[1].legend()
    axes[1].axis('off')

    # 3) Google Gemini (bounding boxy)
    axes[2].imshow(img_rgb)
    # Pink box
    x_min, y_min, x_max, y_max = xy_array_gemini_bbox[nr_xy][0:4]
    rect1 = patches.Rectangle((x_min*w, y_min*h), (x_max-x_min)*w, (y_max-y_min)*h, 
                               linewidth=2, edgecolor='magenta', facecolor='none', label='Pink')
    axes[2].add_patch(rect1)
    # Green box
    x_min, y_min, x_max, y_max = xy_array_gemini_bbox[nr_xy][4:8]
    rect2 = patches.Rectangle((x_min*w, y_min*h), (x_max-x_min)*w, (y_max-y_min)*h, 
                               linewidth=2, edgecolor='green', facecolor='none', label='Green')
    axes[2].add_patch(rect2)
    # Blue box
    x_min, y_min, x_max, y_max = xy_array_gemini_bbox[nr_xy][8:12]
    rect3 = patches.Rectangle((x_min*w, y_min*h), (x_max-x_min)*w, (y_max-y_min)*h, 
                               linewidth=2, edgecolor='blue', facecolor='none', label='Blue')
    axes[2].add_patch(rect3)
    axes[2].set_title('Google Gemini (BBox)', fontsize=16)
    axes[2].legend()
    axes[2].axis('off')

    plt.tight_layout()
    out = plots_bbox_dir / f"comparison_bbox_{nr_xy:02d}.png"
    plt.savefig(out, dpi=300, bbox_inches='tight')
    print(f"Zapisano: {out}")
    plt.show()

print(f"\nWszystkie ploty bbox zostały zapisane w folderze: {plots_bbox_dir}")

# Fine-tune konfiguracja robota

In [ ]:
import pandas as pd
from pathlib import Path
import base64
import numpy as np

In [ ]:
SAVE_DIR = Path("outputs")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

incomplete_detect = np.load(str(SAVE_DIR)+'/incomplete_detect.npy')
df = pd.read_csv(str(SAVE_DIR)+'/df.csv')
incomplete_detect

In [ ]:
df = df.drop(incomplete_detect)

In [ ]:
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

In [ ]:
df["color_path"] = df["color_path"].apply(encode_image)
df = df.drop(columns=["depth_path", "time", "id"])

In [ ]:
df

In [ ]:
from openai import OpenAI
import os

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [ ]:
from sklearn.model_selection import train_test_split
import json

output_columns = [
    "ESJoint1", "ESJoint2", "ESJoint3", "ESJoint4", "ESJoint5", "ESJoint6",
    "gripper_finger_1_joint", "gripper_finger_2_joint"
]

# Podział na train/val/test
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=0)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=0)

def save_jsonl_text(dataframe, filename):
    with open(filename, "w") as f:
        for idx, row in dataframe.iterrows():
            user_content = f"Obraz chwytaka robota (base64): {row['color_path']}\nPodaj konfigurację robota."
            config = ", ".join(f"{col}: {row[col]}" for col in output_columns)
            assistant_content = f"Konfiguracja robota: {config}"
            example = {
                "messages": [
                    {"role": "user", "content": user_content},
                    {"role": "assistant", "content": assistant_content}
                ]
            }
            f.write(json.dumps(example, ensure_ascii=False) + "\n")

save_jsonl_text(train_df, "train_text.jsonl")
save_jsonl_text(val_df, "val_text.jsonl")
save_jsonl_text(test_df, "test_text.jsonl")

In [ ]:
from sklearn.model_selection import train_test_split
import json
import base64

output_columns = [
    "ESJoint1", "ESJoint2", "ESJoint3", "ESJoint4", "ESJoint5", "ESJoint6",
    "gripper_finger_1_joint", "gripper_finger_2_joint"
]

# Przygotowanie danych do Vision fine-tuning
# Wczytaj dane ponownie bez kodowania base64
df_vision = pd.read_csv(str(SAVE_DIR)+'/df.csv')
df_vision = df_vision.drop(incomplete_detect)
df_vision = df_vision.drop(columns=["depth_path", "time", "id"])

# Podział na train/val/test dla Vision
train_df_vision, temp_df_vision = train_test_split(df_vision, test_size=0.3, random_state=0)
val_df_vision, test_df_vision = train_test_split(temp_df_vision, test_size=0.5, random_state=0)

def encode_image_base64(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

def save_jsonl_vision(dataframe, filename):
    with open(filename, "w") as f:
        for idx, row in dataframe.iterrows():
            # Koduj obraz do base64
            base64_image = encode_image_base64(row['color_path'])
            config = ", ".join(f"{col}: {row[col]}" for col in output_columns)
            example = {
                "messages": [
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": "Podaj konfigurację robota na podstawie obrazu chwytaka."},
                            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}}
                        ]
                    },
                    {
                        "role": "assistant",
                        "content": f"Konfiguracja robota: {config}"
                    }
                ]
            }
            f.write(json.dumps(example, ensure_ascii=False) + "\n")
            if idx % 10 == 0:
                print(config)

# Generuj pliki JSONL dla Vision fine-tuning
save_jsonl_vision(train_df_vision, "train_vision.jsonl")
save_jsonl_vision(val_df_vision, "val_vision.jsonl")
save_jsonl_vision(test_df_vision, "test_vision.jsonl")

print("Pliki JSONL dla Vision fine-tuning zostały utworzone.")
print(f"Train: {len(train_df_vision)} przykładów")
print(f"Val: {len(val_df_vision)} przykładów") 
print(f"Test: {len(test_df_vision)} przykładów")

In [ ]:
# Przygotowanie plików dla OpenAI Vision fine-tuning
import shutil
import zipfile

# Utwórz katalog dla obrazów  
images_dir = Path("vision_finetune_images")
images_dir.mkdir(exist_ok=True)

# Skopiuj wszystkie obrazy do jednego katalogu
all_image_paths = set()
for df_split in [train_df_vision, val_df_vision, test_df_vision]:
    for _, row in df_split.iterrows():
        src_path = row['color_path']
        dst_path = images_dir / Path(src_path).name
        if not dst_path.exists():  # Skopiuj tylko jeśli nie istnieje
            shutil.copy2(src_path, dst_path)
        all_image_paths.add(src_path)

print(f"Skopiowano {len(all_image_paths)} unikalnych obrazów do {images_dir}")

# Utwórz kompletne archiwum ZIP z obrazami i plikami JSONL
with zipfile.ZipFile("vision_finetune_complete.zip", 'w') as zipf:
    # Dodaj pliki JSONL
    zipf.write("train_vision.jsonl")
    zipf.write("val_vision.jsonl") 
    zipf.write("test_vision.jsonl")
    
    # Dodaj wszystkie obrazy do katalogu "images/" w ZIP
    for image_file in images_dir.glob("*"):
        if image_file.is_file():
            zipf.write(image_file, f"images/{image_file.name}")

print("Utworzono vision_finetune_complete.zip")

In [ ]:
# Upload plików do OpenAI i uruchomienie fine-tuningu przez Python API
import os
from openai import OpenAI

# Inicjalizacja klienta OpenAI
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

try:
    # Upload training file
    print("Przesyłanie pliku treningowego...")
    with open("train_vision.jsonl", "rb") as f:
        train_file = client.files.create(file=f, purpose="fine-tune")
    print(f"Plik treningowy przesłany: {train_file.id}")

    # Upload validation file
    print("Przesyłanie pliku walidacyjnego...")
    with open("val_vision.jsonl", "rb") as f:
        val_file = client.files.create(file=f, purpose="fine-tune")
    print(f"Plik walidacyjny przesłany: {val_file.id}")

    # Uruchom fine-tuning
    print("Uruchamianie fine-tuning...")
    fine_tune_job = client.fine_tuning.jobs.create(
        training_file=train_file.id,
        validation_file=val_file.id,
        model="gpt-4o-2024-08-06",
        suffix="robot-config"
    )

    print(f"Fine-tuning job uruchomiony!")
    print(f"Job ID: {fine_tune_job.id}")
    print(f"Status: {fine_tune_job.status}")
    print(f"Model: {fine_tune_job.model}")
    
except Exception as e:
    print(f"Błąd: {e}")
    print("Sprawdź czy:")
    print("1. Zmienna OPENAI_API_KEY jest ustawiona")
    print("2. Pliki train_vision.jsonl i val_vision.jsonl istnieją")
    print("3. Masz wystarczający balans na koncie OpenAI")

In [ ]:
# Wczytaj oryginalne dane
df = pd.read_csv("outputs/df.csv")
incomplete_detect = np.load("outputs/incomplete_detect.npy")
df_vision = df.drop(incomplete_detect).drop(columns=["depth_path", "time", "id"])

# Odtwórz podział z tym samym random_state
train_df_vision, temp_df_vision = train_test_split(df_vision, test_size=0.3, random_state=0)
val_df_vision, test_df_vision = train_test_split(temp_df_vision, test_size=0.5, random_state=0)

# Pokaż nazwy plików testowych
test_image_names = [Path(path).name for path in test_df_vision['color_path']]
print("Obrazy testowe:")
for name in test_image_names:
    print(name)